<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/notebooks/5_Ensemble_NVT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula Prática 05 - Dinâmica Molecular em Ensemble NVT

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## Dinâmica Molecular em Ensemble NVT

- $N$: número de partículas
- $V$: volume
- $T$: temperatura

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.animation import PillowWriter

## Integrador de Langevin

A dinâmica de Langevin é dada por

$$dx = \frac{p}{m} dt$$
$$dp = - \nabla U(x) dt - \gamma p dt + \left(2 \gamma k_B T m\right)^{1/2} dW$$

onde $\gamma$ é um coeficiente de atrito e $dW$ é um processo estocástico gerado a partir de um número aleatório gaussiano com média

$$\langle dW \rangle = 0$$

e variância

$$\langle dW^2 \rangle = dt$$

Função que calcula o potencial de LJ entre pares

In [ ]:
def ulj(r,epsilon=1.0,sigma=1.0):
    return 4*epsilon*((sigma/r)**12-(sigma/r)**6)

Função que calcula a derivada do potencial de LJ entre pares

In [ ]:
def duljdr(r,epsilon=1.0,sigma=1.0):
    return -48*epsilon*((sigma/r)**12-0.5*(sigma/r)**6)/r


> **<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**
> 
> Implemente o integrador de langevin usando o procedimento BAOA
> 
> Ref: https://arxiv.org/abs/2204.02105


In [ ]:
class LJsystem():
    # Função que cria a classe
    def __init__(self,N, m = 1.0, epsilon = 1.0, sigma = 1.0, rcut = 2.5, dt = 0.003, integrator='verlet'):
        # número de partículas
        self.N = N

        # passo de tempo
        self.dt = dt

        # integrador de equações de movimento
        self.integrator = integrator

        # parametros dos atomos
        self.m = m
        self.sigma = sigma
        self.epsilon = epsilon

        # parametro de Langevin
        self.gamma = 0.3

        # parametro de cutoff da interação de LJ
        self.rcut = rcut

        # Arrays de posições
        self.r = np.zeros((self.N,3))

        # Arrays de velocidade
        self.v = np.zeros((self.N,3))

        # Arrays de aceleração
        self.a = np.zeros((self.N,3))

    def set_density(self,rho):
        self.rho = rho
        # calcular o tamanho da caixa necessária
        self.Vol = self.N/self.rho
        self.L = np.power(self.Vol,1/3.0)

    def initialize_positions(self):
        # Número de pontos em x e y
        Nx = Ny = Nz = int(np.ceil(np.power(self.N/4,1/3.0)))

        # Espaçamento entre os átomos
        dx = self.L/Nx
        dy = self.L/Ny
        dz = self.L/Nz

        # Posições possíveis em x e y numa rede cristalina tipo FCC
        id = 0
        for i in range(Nx):
            for j in range(Ny):
                for k in range(Nz):
                    if (id < self.N):
                        self.r[id][0] = i*dx
                        self.r[id][1] = j*dy
                        self.r[id][2] = k*dz
                        id +=1

                    if (id < self.N):
                        self.r[id][0] = i*dx
                        self.r[id][1] = (j+0.5)*dy
                        self.r[id][2] = (k+0.5)*dz
                        id +=1

                    if (id < self.N):
                        self.r[id][0] = (i+0.5)*dx
                        self.r[id][1] = j*dy
                        self.r[id][2] = (k+0.5)*dz
                        id +=1

                    if (id < self.N):
                        self.r[id][0] = (i+0.5)*dx
                        self.r[id][1] = (j+0.5)*dy
                        self.r[id][2] = k*dz
                        id +=1

    def initialize_velocities(self,kT= 1.0):
        self.kT = kT

        # sorteia uma distribuição normal com devio padrão proporcional a temperatura
        self.v = np.sqrt(self.kT/self.m)*np.random.randn(self.N,3)

        # remove o momento linear total
        self.v[:] -= self.v.mean(axis=0)

    def build_neighbor_list(self,rcell=2.8):
        """Construindo uma lista de vizinhos de Verlet dentro de uma região rcell"""
        self.rcell = rcell
        self.NL = [[] for _ in range(self.N)]
        for i in range(self.N):
            for j in range(i+1, self.N):
                # calcula distancia entre duas particulas
                rij = self.r[i]-self.r[j]
                # testa condição de contorno periodica para imagens
                rij[:] -= np.rint(rij/self.L)*self.L
                # calcula modulo da distancia
                rij_norm = np.linalg.norm(rij)
                # Vamos considerar somente interação dentro do raio de corte
                if rij_norm < self.rcell*self.sigma:
                    self.NL[i].append(j)

    def update_forces(self):
        self.a.fill(0.0)
        self.U = 0.0
        self.W = 0.0
        for i in range(self.N):
            for j in self.NL[i]: # <-------- troca os limites
                # calcula distancia entre duas particulas
                rij = self.r[i]-self.r[j]
                # testa condição de contorno periodica para imagens
                rij[:] -= np.rint(rij/self.L)*self.L
                # calcula modulo da distancia
                rij_norm = np.linalg.norm(rij)
                # Vamos considerar somente interação dentro do raio de corte
                if rij_norm < self.rcut*self.sigma:
                    # calcula a derivada do potencial
                    dudr = duljdr(rij_norm,self.epsilon,self.sigma)
                    # calcula aceleração na particula i
                    self.a[i] += -dudr*rij/rij_norm
                    # calcula aceleração na particula j usando 3ª Lei de Newton
                    self.a[j] += dudr*rij/rij_norm
                    # calcula energia interna
                    self.U += ulj(rij_norm,self.epsilon,self.sigma)
                    self.W += dudr*rij_norm

    def initialize(self,kT=1.0):
        self.initialize_positions()
        self.initialize_velocities(kT=kT)
        self.build_neighbor_list()
        self.update_forces()

    def potential_energy(self):
        return self.U

    def kinetic_energy(self):
        return 0.5*self.m*np.sum(self.v**2)

    def temperature(self):
        K = self.kinetic_energy()
        return 2*K/(3*self.N)

    def pressure(self):
        K = self.kinetic_energy()
        kT = self.temperature()
        return self.rho*kT + self.W/(3*self.Vol)

    def thermodynamic_properties(self):
        K = self.kinetic_energy()
        U = self.potential_energy()
        T = self.temperature()
        P = self.pressure()
        E = K + U
        return K, U, E, T, P
    
    def verlet_integrator(self):
        # update das velocidades
        self.v[:] += 0.5*self.a*self.dt
        # update das posições
        self.r[:] += self.dt * self.v
        # condição de contorno periódica
        self.r[:] = self.r % self.L
        # calcula nova aceleração
        self.update_forces()
        # update das velocidades
        self.v[:] += 0.5*self.a*self.dt

    def langevin_integrator(self):
        # implementação do integrador de Langevin usando método BAOA
        ...


    def step(self):
        if self.integrator == 'verlet':
            self.verlet_integrator()
        elif self.integrator == 'langevin':
            self.langevin_integrator()
        else:
            raise ValueError("Integrador desconhecido: escolha 'verlet' ou 'langevin'")


    def run(self,N_steps,print_every=50,update_list=20):
        self.N_steps = N_steps
        self.print_every = print_every # steps to print output
        self.update_list = update_list

        print("iter\tK\tU\tE\tT\tP")
        self.data = pd.DataFrame({'t': [], 'K': [], 'U': [], 'E': [], 'T': [], 'P': []})

        for step in range(1,self.N_steps+1):

            if step % self.update_list == 0:
                self.build_neighbor_list()

            self.step()

            if step % self.print_every == 0:
                K, U, E, T, P = self.thermodynamic_properties()

                print(f"{step:5d}\t{K:.4f}\t{U:.4f}\t{E:.4f}\t{T:.4f}\t{P:.4f}")
                new_row = pd.DataFrame({'t': [step], 'K': [K], 'U': [U], 'E': [E], 'T': [T], 'P': [P]})
                self.data = pd.concat([self.data, new_row], ignore_index=True)

In [ ]:
rho = 0.2
kT = 1.5

N_atoms=256

In [ ]:
lj = LJsystem(N=N_atoms,m=1.0,epsilon=1.0,sigma=1.0, rcut=2.5, dt=0.003, integrator='langevin')
lj.set_density(rho=rho)
lj.initialize(kT=kT)

Gráfico com as posições iniciais das partículas

In [ ]:
plt.figure(figsize=(5,5))
plt.plot(lj.r[:,0],lj.r[:,1],'o',ms=10.0,alpha=0.5) # posição das partículas
plt.quiver(lj.r[:,0],lj.r[:,1],lj.v[:,0],lj.v[:,1],color='C0') # vetor de velocidade

plt.xlim(0,lj.L)
plt.ylim(0,lj.L)

plt.xlabel('x')
plt.ylabel('y')

In [ ]:
lj.run(N_steps=10000, print_every=50, update_list=20)

Gráfico com as posições finais

In [ ]:
plt.figure(figsize=(5,5))
plt.plot(lj.r[:,0],lj.r[:,1],'o',ms=10.0,alpha=0.5) # posição das partículas
plt.quiver(lj.r[:,0],lj.r[:,1],lj.v[:,0],lj.v[:,1],color='C0') # vetor de velocidade

plt.xlim(0,lj.L)
plt.ylim(0,lj.L)

plt.xlabel('x')
plt.ylabel('y')

Gerar arquivo de output no formato `.txt` com qtds de interesse

In [ ]:
simdata = lj.data

# Save as tab-separated .txt file
simdata.to_csv(f'simdata-N={N_atoms}-rho={rho:.3f}-kT={kT:.3f}-Nsteps={lj.N_steps}.txt', sep='\t', index=False)

Distribuição espacial das partículas

In [ ]:
fig, axs = plt.subplots(1, 3, sharey=True,figsize=(10,3))

axs[0].hist(lj.r[:,0],bins=10,range=(0,lj.L),density=True,color='C0',alpha=0.5)
axs[0].hlines(y=rho*lj.L**2/lj.N,xmin=0,xmax=lj.L,color='k')
axs[0].set_xlabel(r'$x/\sigma$')
axs[0].set_ylabel(r'$\rho/\sigma^3$')

axs[1].hist(lj.r[:,1],bins=10,range=(0,lj.L),density=True,color='C1',alpha=0.5)
axs[1].hlines(y=rho*lj.L**2/lj.N,xmin=0,xmax=lj.L,color='k')
axs[1].set_xlabel(r'$y/\sigma$')

axs[2].hist(lj.r[:,2],bins=10,range=(0,lj.L),density=True,color='C2',alpha=0.5)
axs[2].hlines(y=rho*lj.L**2/lj.N,xmin=0,xmax=lj.L,color='k')
axs[2].set_xlabel(r'$z/\sigma$')

axs[0].set_ylim(0,0.2)

Distribuição de Velocidades

In [ ]:
fig, axs = plt.subplots(1, 3, sharey=True,figsize=(10,3))

# cria uma distribuição gaussiana de velocidades em x
# com sigma^2 = kT/m
vxarray = np.arange(-5.0,5.0,0.01)*np.sqrt(kT)
fvx = np.exp(-0.5*vxarray**2/kT)/np.sqrt(2*np.pi*kT)

axs[0].hist(lj.v[:,0],bins=40,range=(vxarray.min(),vxarray.max()),density=True,color='C0',alpha=0.5)
axs[0].plot(vxarray,fvx,'k')
axs[0].set_xlabel(r'$v_x$')

axs[1].hist(lj.v[:,1],bins=40,range=(vxarray.min(),vxarray.max()),density=True,color='C1',alpha=0.5)
axs[1].plot(vxarray,fvx,'k')
axs[1].set_xlabel(r'$v_y$')

axs[2].hist(lj.v[:,2],bins=40,range=(vxarray.min(),vxarray.max()),density=True,color='C2',alpha=0.5)
axs[2].plot(vxarray,fvx,'k')
axs[2].set_xlabel(r'$v_z$')

axs[0].set_ylim(0,0.5)

Gráfico da evolução de $E$, $U$ e $T$ como função dos passos de iteração

In [ ]:
fig, axs = plt.subplots(4, 1, sharex=True)

axs[0].plot(simdata['t'],simdata['E']/lj.N,'k',label='E')
axs[0].legend(loc='best')
axs[0].set_ylabel(r'$E/(N\epsilon)$')

axs[1].plot(simdata['t'],simdata['U']/lj.N,color='C0',label='U')
axs[1].legend(loc='best')
axs[1].set_ylabel(r'$U/(N\epsilon)$')

axs[2].plot(simdata['t'],simdata['P'],color='C1',label='P')
axs[2].legend(loc='best')
axs[2].set_ylabel(r'$P\sigma^3/\epsilon$')

axs[3].plot(simdata['t'],simdata['T'],color='C3',label='T')
axs[3].legend(loc='best')
axs[3].set_xlabel(r'$t/\tau$')
axs[3].set_ylabel(r'$k_B T/\epsilon$')

## Análise das distribuições de energia interna, temperatura e pressão.


Cortando os passos em que o sistema não está equilibrado

In [ ]:
teq = 2000
subsett = simdata['t'][simdata['t']>teq]
subsetT = simdata['T'][simdata['t']>teq]
subsetP = simdata['P'][simdata['t']>teq]
subsetU = simdata['U'][simdata['t']>teq]/lj.N

> **<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**
>
> Determine a média e o desvio padrão da temperatura $kT$ e faça o gráfico da distribuição dos dados

In [ ]:
Tmean = ...
Tstd = ...

print(f"kT: {Tmean:.3f} ± {Tstd:.3f}")

In [ ]:
x = np.arange(-5.0,5.0,0.01)*Tstd + Tmean # temperatura

plt.hist(subsetT,bins=20,range=(x.min(),x.max()),density=True,color='C3',alpha=0.5)

# faça o gráfico da distribuição teórica gaussiana de temperatura
plt.plot(...)

plt.xlabel('kT')

> **<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**
>
> Determine a média e o desvio padrão da energia interna $U/N$ e faça o gráfico da distribuição dos dados

In [ ]:
Umean = ...
Ustd = ...

print(f"U/N: {Umean:.3f} ± {Ustd:.3f}")

In [ ]:
x = np.arange(-5.0,5.0,0.01)*Ustd + Umean # energia potencial por partícula

plt.hist(subsetU,bins=20,range=(x.min(),x.max()),density=True,color='C0',alpha=0.5)

# faça o gráfico da distribuição teórica gaussiana
plt.plot(...)

plt.xlabel('U/N')

> **<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**
>
> Determine a média e o desvio padrão da pressão $P$ e faça o gráfico da distribuição dos dados

In [ ]:
Pmean = ...
Pstd = ...

print(f"P: {Pmean:.3f} ± {Pstd:.3f}")

In [ ]:
x = np.arange(-5.0,5.0,0.01)*Pstd + Pmean # pressão

plt.hist(subsetP,bins=20,range=(x.min(),x.max()),density=True,color='C1',alpha=0.5)

# faça o gráfico da distribuição teórica gaussiana 
plt.plot(...)

plt.xlabel('P')